# 🏥 Pre-Consultation Agent — Full Pipeline Testing (Google Colab)

This notebook tests the **complete two-stage extraction system** with intelligent routing on Colab's **free T4 GPU**.

## What This Tests:
- ✅ **Model A**: Audio transcription + quality assessment
- ✅ **Model B Light**: Quick extraction for routing
- ✅ **Conversation Router**: Emergency / Rule-based / AI-powered paths
- ✅ **Model C Rules**: Predefined questions for known symptoms
- ✅ **Model C AI**: Adaptive questions for complex cases
- ✅ **Patient Info Collection**: Name, age, gender
- ✅ **Model B Full**: Comprehensive extraction after conversation
- ✅ **Models D–F**: Risk scoring, patient guidance, doctor summary

## Setup Checklist (do this BEFORE running any cell):
1. ✅ Enable GPU: **Runtime → Change runtime type → T4 GPU → Save**
2. ✅ Add Secrets (🔑 key icon in the left sidebar):
   - `GEMINI_API_KEY` — get from [Google AI Studio](https://aistudio.google.com/apikey)
   - `HF_TOKEN` — get from [Hugging Face settings](https://huggingface.co/settings/tokens)
   - `NGROK_AUTH_TOKEN` — get from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
   - *(Optional)* `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD` — for kiosk API endpoints
3. ✅ Make sure **"Notebook access"** is enabled for each secret

**Free GPU quota**: ~12 hours per session, resets every 24 hours.

Let's go! 🚀

## Step 1: Verify GPU and Environment

In [ ]:
import torch
import os
import sys

print("🔍 Checking environment...\n")

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU detected!")
    print("   Go to: Runtime → Change runtime type → T4 GPU → Save")
    print("   Then reconnect and re-run this cell.")

print(f"\n🐍 Python: {sys.version.split()[0]}")
print(f"📁 Working dir: {os.getcwd()}")

## Step 2: Clone / Update Repository

In [ ]:
import os
from pathlib import Path

repo_path = Path('/content/pre-consultation-agent')

if repo_path.exists():
    print("🔄 Repository already cloned — pulling latest changes...\n")
    os.chdir(str(repo_path))
    !git pull origin main
    print("\n✅ Latest code pulled!")
else:
    print("📥 Cloning repository...\n")
    !git clone https://github.com/buwituze/pre-consultation-agent.git /content/pre-consultation-agent
    print("\n✅ Repository cloned!")

os.chdir('/content/pre-consultation-agent/backend')
print(f"\n📁 Current directory: {os.getcwd()}")

## Step 3: Install Dependencies

In [ ]:
print("📦 Installing dependencies...\n")
!pip install -q -r requirements.txt

print("✅ Dependencies installed!\n")

# Verify key modules
import importlib
for pkg in ['transformers', 'google.genai', 'psycopg2', 'pydantic', 'fastapi']:
    try:
        m = importlib.import_module(pkg)
        version = getattr(m, '__version__', 'installed')
        print(f"   ✅ {pkg}: {version}")
    except ImportError:
        print(f"   ❌ {pkg}: NOT FOUND")

## Step 4: Configure API Keys

Reads secrets from the Colab **Secrets panel** (🔑 icon in the left sidebar).
Each secret must have **"Notebook access" toggled ON** before running this cell.

In [ ]:
import os
from google.colab import userdata

# ── Helper: try secret, fall back to placeholder ──────────────────────────────
def get_secret(name, fallback=None):
    try:
        return userdata.get(name)
    except Exception:
        return fallback or f'YOUR_{name}_HERE'

# ── Model / API keys ──────────────────────────────────────────────────────────
gemini_key = get_secret('GEMINI_API_KEY')
hf_token   = get_secret('HF_TOKEN')

os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['HF_TOKEN']       = hf_token
os.environ['DEVICE']         = 'cuda'
os.environ['MAX_TURNS']      = '10'
os.environ['SKIP_LANG_DETECTION_WHEN_HINTED'] = 'true'

print("✅ Environment configured:")
print(f"   GEMINI_API_KEY : {'✓ SET' if 'YOUR_' not in gemini_key else '❌ NOT SET — add it in the 🔑 panel'}")
print(f"   HF_TOKEN       : {'✓ SET' if 'YOUR_' not in hf_token   else '❌ NOT SET — add it in the 🔑 panel'}")
print(f"   DEVICE         : cuda (GPU)")

# ── Database (optional — needed only for kiosk API endpoints) ────────────────
db_host = get_secret('DB_HOST', 'localhost')
db_port = get_secret('DB_PORT', '5432')
db_name = get_secret('DB_NAME', 'pre_consultation_db')
db_user = get_secret('DB_USER', 'postgres')
db_pass = get_secret('DB_PASSWORD', '')

os.environ['DB_HOST']     = db_host
os.environ['DB_PORT']     = db_port
os.environ['DB_NAME']     = db_name
os.environ['DB_USER']     = db_user
os.environ['DB_PASSWORD'] = db_pass
os.environ['USE_DB']      = 'true'

print(f"\n🔧 Database config: {db_host}:{db_port} / {db_name} (user: {db_user})")

# Quick connection test
try:
    import psycopg2
    conn = psycopg2.connect(
        host=db_host, port=db_port,
        database=db_name, user=db_user, password=db_pass,
        connect_timeout=5,
    )
    conn.close()
    print("✅ Database connection OK")
except Exception as e:
    print(f"⚠️  Database not reachable ({e})")
    print("   Pipeline tests will still work — only kiosk API endpoints need a DB.")

## Step 5: Upload Audio File

Run this cell to upload a `.wav` file from your local machine.  
The file is stored under `/content/audio/` and its path is saved in `audio_path`.

> **Tip**: You can also mount Google Drive and reference a file already there:
> ```python
> from google.colab import drive
> drive.mount('/content/drive')
> audio_path = '/content/drive/MyDrive/your_audio.wav'
> ```

In [ ]:
import os
from google.colab import files

os.makedirs('/content/audio', exist_ok=True)

print("📤 Select your WAV file to upload...")
uploaded = files.upload()

wav_files = []
for filename, data in uploaded.items():
    dest = f'/content/audio/{filename}'
    with open(dest, 'wb') as f:
        f.write(data)
    wav_files.append(dest)
    print(f"✅ Saved: {dest} ({len(data)/(1024*1024):.2f} MB)")

if not wav_files:
    print("❌ No file uploaded. Re-run this cell and choose a .wav file.")
else:
    audio_path = wav_files[0]
    print(f"\n🎤 Will use: {audio_path}")

## Step 6: Load Whisper Models (Model A)

First run downloads ~6 GB of model weights.  
Subsequent runs in the same session load them from the Colab instance cache (~30 s).

In [ ]:
import sys
import time
import torch

sys.path.insert(0, '/content/pre-consultation-agent/backend')

from models import model_a

print("🔄 Loading Whisper models on GPU...")
print("   First run: 3–5 minutes (downloading ~6 GB)\n")

start = time.time()
model_a.load_models()
elapsed = time.time() - start

print(f"\n✅ Models loaded in {elapsed:.1f}s")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"📊 GPU Memory: {allocated:.2f} GB / {total:.1f} GB used")

## Step 7: Full Conversation Test — Phase 1 (Transcription + Routing)

**Phase 1 (auto):** Audio → Transcription → Light Extraction → Routing Decision

In [ ]:
# ============================================================
# PHASE 1: Audio → Transcription → Light Extraction → Routing
# ============================================================
import time
import os
from models import model_a, model_b, model_c, model_d, model_e, model_f
from models.model_c_rules import get_symptom_questions, get_patient_info_questions
from routing.conversation_router import route_conversation

print("=" * 70)
print("🏥 PHASE 1: TRANSCRIPTION + ROUTING")
print("=" * 70)

# Make sure audio_path is defined (from Step 5)
if 'audio_path' not in dir() or not os.path.exists(audio_path):
    raise RuntimeError("audio_path is not set — run Step 5 to upload your audio file first.")

print(f"\n🎤 Audio: {os.path.basename(audio_path)} ({os.path.getsize(audio_path)/(1024*1024):.2f} MB)\n")

with open(audio_path, 'rb') as f:
    audio_bytes = f.read()

# --- Model A: Transcription ---
print("─" * 70)
print("MODEL A — Transcription + Quality Assessment")
print("─" * 70)

start = time.time()
transcription = model_a.transcribe(audio_bytes, language_hint='kinyarwanda')
elapsed = time.time() - start

transcript  = transcription['full_text']
language    = transcription['dominant_language']
confidence  = transcription['mean_confidence']
quality     = transcription.get('quality', 'medium')

print(f"✅ Complete in {elapsed:.1f}s")
print(f"   Language: {language} | Confidence: {confidence:.0%} | Quality: {quality}")
print(f"\n📝 Transcript:\n   {transcript}\n")

# --- Model B Light: Quick Extraction ---
print("─" * 70)
print("MODEL B LIGHT — Extract Routing Info")
print("─" * 70)

start = time.time()
light_extraction = model_b.extract_light(transcript)
elapsed = time.time() - start

chief_complaint = light_extraction.get('chief_complaint', 'unknown')
severity        = light_extraction.get('severity_estimate', 0)
red_flags       = light_extraction.get('red_flags_present', False)

print(f"✅ Complete in {elapsed:.1f}s")
print(f"   Chief Complaint: {chief_complaint}")
print(f"   Severity: {severity}/10")
print(f"   Red Flags: {'🚨 YES' if red_flags else '✅ No'}\n")

# --- Conversation Router ---
print("─" * 70)
print("ROUTER — Decide Conversation Path")
print("─" * 70)

routing = route_conversation(
    light_extraction=light_extraction,
    transcription_quality=quality,
    language=language
)

mode      = routing['mode']
reasoning = routing['reasoning']

print(f"🎯 Mode: {mode.upper()}")
print(f"   Reasoning: {reasoning}")

lang_key       = 'kinyarwanda' if language == 'kinyarwanda' else 'english'
preset_questions = get_symptom_questions(chief_complaint, lang_key)

if mode == 'emergency':
    print("\n🚨 EMERGENCY — patient will be fast-tracked with minimal questions")
elif mode == 'rule_based' and preset_questions:
    print(f"\n📋 RULE-BASED — {len(preset_questions)} preset questions for '{chief_complaint}'")
elif mode == 'ai_powered':
    if preset_questions:
        print("\n🤖 AI-POWERED — preset questions exist, will use those first")
    else:
        print("\n🤖 AI-POWERED — no preset questions, Model C will generate them")

print(f"\n{'=' * 70}")
print("✅ Phase 1 complete! Run the next cell for the interactive conversation.")
print(f"{'=' * 70}")

## Step 7b: Phase 2 — Interactive Patient Conversation

**You type answers as if you are the patient.**  
In the real kiosk the patient would speak; here we skip transcription and type directly — the conversation logic being tested is identical.

The system will:
1. Ask patient info questions (name, age, gender) — always first
2. Ask symptom questions based on the routing decision:
   - **Rule-based**: preset questions for known symptoms
   - **AI-powered**: Model C generates adaptive questions
   - **Emergency**: minimal questions, fast-track to scoring

In [ ]:
# ============================================================
# PHASE 2: Interactive Patient Conversation
# ============================================================
print("=" * 70)
print("🏥 PHASE 2: PATIENT CONVERSATION")
print("=" * 70)
print("Type your answers below as if you are the patient.\n")

conversation_turns = []

# --- Step 1: Patient Info (Always First) ---
print("─" * 70)
print("PATIENT INFO — Always asked first")
print("─" * 70)

info_questions = get_patient_info_questions(lang_key)

patient_name   = ""
patient_age    = None
patient_gender = ""

for q_dict in info_questions:
    print(f"\n🤖 System asks: {q_dict['question']}")
    answer = input("👤 Your answer: ").strip()

    if q_dict['id'] == 'patient_name':
        patient_name = answer
    elif q_dict['id'] == 'patient_age':
        try:
            patient_age = int(answer)
        except ValueError:
            patient_age = 30
            print(f"   (Could not parse age, using default: {patient_age})")
    elif q_dict['id'] == 'patient_gender':
        patient_gender = answer

    conversation_turns.append({'question': q_dict['question'], 'answer': answer})

print(f"\n✅ Patient info collected: {patient_name}, age {patient_age}, {patient_gender}")

# --- Step 2: Symptom Questions (Based on Routing) ---
print(f"\n{'─' * 70}")
print(f"SYMPTOM QUESTIONS — {mode.upper()} PATH")
print("─" * 70)

symptom_turns = []

if mode == 'emergency':
    eq = "Can you describe your main concern right now?" if lang_key == 'english' else "Ni iki gikomeye ubu?"
    print("\n🚨 EMERGENCY — one quick question before fast-tracking\n")
    print(f"🤖 System asks: {eq}")
    answer = input("👤 Your answer: ").strip()
    symptom_turns.append({'question': eq, 'answer': answer})

elif mode == 'rule_based' and preset_questions:
    print(f"\n📋 Using {len(preset_questions)} preset questions for '{chief_complaint}'\n")
    for i, q_dict in enumerate(preset_questions, 1):
        print(f"\n🤖 Q{i}: {q_dict['question']}")
        answer = input("👤 Your answer: ").strip()
        symptom_turns.append({'question': q_dict['question'], 'answer': answer})

else:
    max_questions = 4
    print(f"\n🤖 Model C will generate up to {max_questions} adaptive questions\n")

    for i in range(1, max_questions + 1):
        try:
            question = model_c.select_next_question(
                extraction=light_extraction,
                questions_asked=[t['question'] for t in symptom_turns],
                patient_answers=[t['answer']   for t in symptom_turns]
            )
            print(f"\n🤖 AI Q{i}: {question}")
            answer = input("👤 Your answer: ").strip()
            symptom_turns.append({'question': question, 'answer': answer})

            combined = dict(light_extraction)
            if model_c.is_coverage_complete(combined, len(symptom_turns), int(os.environ.get('MAX_TURNS', 10))):
                print(f"\n✅ Coverage complete after {len(symptom_turns)} questions")
                break
        except Exception as e:
            print(f"\n⚠️ Model C error: {e}")
            break

conversation_turns.extend(symptom_turns)

print(f"\n{'=' * 70}")
print("✅ Conversation complete!")
print(f"   Patient info questions : {len(info_questions)}")
print(f"   Symptom questions      : {len(symptom_turns)}")
print(f"   Total turns            : {len(conversation_turns)}")
print(f"\n📝 Full conversation log:")
for i, turn in enumerate(conversation_turns, 1):
    print(f"   {i}. Q: {turn['question']}")
    print(f"      A: {turn['answer']}")
print(f"\n{'=' * 70}")
print("Run the next cell for extraction, scoring, and results.")
print(f"{'=' * 70}")

## Step 7c: Phase 3 — Full Extraction + Scoring + Results

Runs the remaining models using the conversation data from Phase 2:
- **Model B Full**: Comprehensive extraction with full conversation context
- **Model D**: Risk scoring and priority assignment
- **Model E**: Patient guidance message
- **Model F**: Doctor clinical summary

In [ ]:
# ============================================================
# PHASE 3: Full Extraction + Scoring + Results
# ============================================================
import time

print("=" * 70)
print("🏥 PHASE 3: EXTRACTION + SCORING + RESULTS")
print("=" * 70)

# --- Model B Full: Comprehensive Extraction ---
print("\n" + "─" * 70)
print("MODEL B FULL — Comprehensive Extraction (with conversation context)")
print("─" * 70)

start = time.time()
try:
    full_extraction = model_b.extract_full(
        transcript=transcript,
        conversation_history=conversation_turns,
        target_language=language
    )
    elapsed = time.time() - start
    print(f"✅ Complete in {elapsed:.1f}s")
    print("\n📊 Extracted Clinical Data:")
    for key, value in full_extraction.items():
        if value and value != 'unknown':
            print(f"   {key}: {value}")
except Exception as e:
    print(f"❌ Full extraction failed: {e}")
    full_extraction = light_extraction

# --- Model D: Risk Scoring ---
print("\n" + "─" * 70)
print("MODEL D — Risk Scoring")
print("─" * 70)

start = time.time()
try:
    risk_score = model_d.score(full_extraction, age=patient_age)
    elapsed = time.time() - start
    print(f"✅ Complete in {elapsed:.1f}s")
    print("\n🎯 Risk Assessment:")
    print(f"   Priority        : {risk_score.get('priority', 'N/A')}")
    print(f"   Suspected Issue : {risk_score.get('suspected_issue', 'N/A')}")
    print(f"   Risk Factors    : {risk_score.get('risk_factors', [])}")
    print(f"   Confidence      : {risk_score.get('confidence', 'N/A')}")
except Exception as e:
    print(f"❌ Risk scoring failed: {e}")
    risk_score = {'priority': 'MEDIUM', 'suspected_issue': 'general complaint', 'risk_factors': [], 'confidence': 0.5}

# --- Model E: Patient Guidance ---
print("\n" + "─" * 70)
print("MODEL E — Patient Guidance Message")
print("─" * 70)

start = time.time()
try:
    patient_guidance = model_e.generate_message(
        extraction=full_extraction,
        score=risk_score,
        language=language,
        location='Waiting Area A'
    )
    elapsed = time.time() - start
    print(f"✅ Complete in {elapsed:.1f}s")
    print(f"\n💬 Message to Patient:\n   {patient_guidance}")
except Exception as e:
    print(f"❌ Patient guidance failed: {e}")
    patient_guidance = 'Please wait to see the doctor.'

# --- Model F: Doctor Summary ---
print("\n" + "─" * 70)
print("MODEL F — Doctor Clinical Summary")
print("─" * 70)

start = time.time()
try:
    doctor_summary = model_f.generate_brief(
        session_id='test-session-001',
        extraction=full_extraction,
        score=risk_score,
        questions_asked=[t['question'] for t in conversation_turns],
        patient_answers=[t['answer']   for t in conversation_turns],
        transcript=transcript,
        language=language,
        patient_age=patient_age
    )
    elapsed = time.time() - start
    print(f"✅ Complete in {elapsed:.1f}s")
    print("\n👨‍⚕️ Doctor Summary:")
    for key, value in doctor_summary.items():
        print(f"   {key}: {value}")
except Exception as e:
    print(f"❌ Doctor summary failed: {e}")
    doctor_summary = {}

# --- Final Summary ---
print("\n" + "=" * 70)
print("📊 SESSION COMPLETE")
print("=" * 70)
print(f"\n🎯 Routing  : {mode.upper()} ({reasoning})")
print(f"👤 Patient  : {patient_name}, age {patient_age}, {patient_gender}")
print(f"🩺 Complaint: {chief_complaint} (severity {severity}/10)")
print(f"⚡ Priority  : {risk_score.get('priority', 'N/A')}")
print(f"📋 Turns    : {len(conversation_turns)}")
print(f"\n{'=' * 70}")
print("✅ FULL PIPELINE TEST COMPLETE (Model A → F)")
print(f"{'=' * 70}")

## Step 8: Fix Invalid Password Hashes (Run Once If Login Returns 500)

If a user was inserted into the database via raw SQL with a **plain-text password**, `passlib` throws `UnknownHashError` and the login endpoint crashes with HTTP 500.

Run the cell below **once** to detect and re-hash any invalid passwords before starting the server.

In [ ]:
import sys, os
import psycopg2
from passlib.context import CryptContext

sys.path.insert(0, '/content/pre-consultation-agent/backend')
pwd_context = CryptContext(schemes=['bcrypt'], deprecated='auto')

for table in ('users', 'app_user'):
    try:
        conn = psycopg2.connect(
            host=os.environ['DB_HOST'], port=os.environ['DB_PORT'],
            database=os.environ['DB_NAME'], user=os.environ['DB_USER'],
            password=os.environ['DB_PASSWORD'],
        )
        cur = conn.cursor()
        cur.execute(f'SELECT user_id, email, password_hash FROM {table};')
        rows = cur.fetchall()
    except Exception as e:
        print(f"⚠️  Could not query table '{table}': {e}")
        continue

    if not rows:
        print(f"ℹ️  No users found in '{table}'.")
        cur.close(); conn.close()
        continue

    print(f"\n🔍 Checking '{table}' ({len(rows)} user(s)):")
    fixed = 0
    for user_id, email, password_hash in rows:
        is_valid = False
        try:
            pwd_context.identify(password_hash)
            is_valid = True
        except Exception:
            pass

        if not is_valid:
            # Re-hash the plain-text value in place so the user keeps the same password.
            new_hash = pwd_context.hash(password_hash)
            cur.execute(
                f'UPDATE {table} SET password_hash = %s WHERE user_id = %s;',
                (new_hash, user_id),
            )
            print(f"   ⚠️  {email} — plain-text detected → re-hashed in place ✅")
            fixed += 1
        else:
            print(f"   ✅ {email} — hash OK")

    conn.commit()
    cur.close()
    conn.close()

    if fixed:
        print(f"\nDone. Fixed {fixed} user(s) in '{table}'.")
    else:
        print(f"\nAll users in '{table}' already have valid bcrypt hashes.")

## Step 9: Deploy API with ngrok (For Frontend Testing)

Starts the FastAPI server and exposes it via ngrok so your Flutter frontend can connect.

### Before running:
1. Make sure `NGROK_AUTH_TOKEN` is set in your Colab Secrets (🔑 panel → notebook access ON)
2. Ensure Step 4 has already been run so the DB environment is configured

### Usage:
- Copy the **API URL** printed below
- Paste it as `BaseURL` in your Flutter app
- The server stays alive as long as this cell is running

> **Colab note**: Colab sessions time out after ~90 minutes of UI inactivity. The [Colab Alive](https://chromewebstore.google.com/detail/colab-alive/eookkckfbbgnhdgcbfbicoahejkdoele) browser extension can keep the session active during longer testing.

In [ ]:
!pip install -q pyngrok

import os
from google.colab import userdata

try:
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ ngrok auth token loaded from Colab Secrets")
except Exception:
    ngrok_token = 'YOUR_NGROK_AUTH_TOKEN_HERE'
    print("⚠️  NGROK_AUTH_TOKEN not found in Secrets.")
    print("   Add it via the 🔑 panel → toggle Notebook access ON")

os.environ['NGROK_AUTH_TOKEN'] = ngrok_token

if 'YOUR_' in ngrok_token:
    print("\n❌ Replace the placeholder with your actual token before continuing!")
    print("   Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    print("✅ ngrok ready to launch")

In [ ]:
import os
import subprocess
import time
import requests
import threading
from pyngrok import ngrok

def read_output(pipe, label):
    for line in iter(pipe.readline, b''):
        print(f"[{label}] {line.decode().rstrip()}")

ngrok.set_auth_token(os.environ['NGROK_AUTH_TOKEN'])

os.chdir('/content/pre-consultation-agent/backend')

# NOTE: leave USE_DB as configured in Step 4.
# Uncomment the line below ONLY if you explicitly want to disable the database:
# os.environ['USE_DB'] = 'false'

port = 8000
process = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', str(port)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ},
)

threading.Thread(target=read_output, args=(process.stdout, 'STDOUT'), daemon=True).start()
threading.Thread(target=read_output, args=(process.stderr, 'STDERR'), daemon=True).start()

print("⏳ Waiting for FastAPI server to start...")
server_ready = False
for _ in range(60):
    time.sleep(2)
    try:
        if requests.get(f'http://localhost:{port}/health').status_code == 200:
            server_ready = True
            break
    except requests.ConnectionError:
        pass

if not server_ready:
    print("❌ Server did not respond after 120 seconds. Check STDERR output above.")
    process.terminate()
    raise RuntimeError("Server startup timeout")

print("✅ FastAPI server started!\n")

print("🌐 Creating ngrok tunnel...")
try:
    tunnel   = ngrok.connect(port, 'http')
    ngrok_url = tunnel.public_url
    print(f"✅ ngrok tunnel created!")
    print(f"\n🚀 API URL: {ngrok_url}")
    print(f"\n📋 Use this in your Flutter app:")
    print(f"   BaseURL = '{ngrok_url}'")

    time.sleep(2)
    try:
        r = requests.get(f'{ngrok_url}/health', timeout=10)
        if r.status_code == 200:
            print("\n✅ Tunnel is working — API accessible from the internet!")
        else:
            print(f"\n⚠️  Tunnel created but health check returned HTTP {r.status_code}")
    except Exception as e:
        print(f"\n⚠️  Could not test tunnel: {e}")

except Exception as e:
    print(f"❌ ngrok tunnel failed: {e}")
    print("\nCommon causes:")
    print("  1. Invalid NGROK_AUTH_TOKEN")
    print("  2. ngrok account tunnel quota exceeded")
    print("  3. Network connectivity issue in this Colab session")
    raise

print(f"\n{'=' * 70}")
print("⏳ Server running... interrupt the cell (⏹) to shut down.")
print(f"{'=' * 70}")

try:
    process.wait()
except KeyboardInterrupt:
    print("\n\n🛑 Shutting down...")
    process.terminate()
    process.wait()
    ngrok.kill()
    print("✅ Shutdown complete")

## Database Notes — Connecting Colab to Your PostgreSQL

### Does the pipeline work without a database?

**Yes.** Steps 1–8 (Models A → F) require **no database at all** — everything runs in memory.

However, the **kiosk API endpoints** that the Flutter frontend calls (`/kiosk/start`, `/kiosk/finish`) **do require a database** because they persist patient records and session data to PostgreSQL.

---

### Option 1: Expose your local DB via ngrok TCP *(recommended for testing)*

On your **local machine** (where PostgreSQL is running):
```bash
ngrok tcp 5432
# → Forwarding  tcp://0.tcp.ngrok.io:XXXXX
```
Then set these Colab Secrets:
```
DB_HOST     = 0.tcp.ngrok.io
DB_PORT     = XXXXX        # the port ngrok assigned
DB_NAME     = pre_consultation_db
DB_USER     = postgres
DB_PASSWORD = your_password
```

### Option 2: Free cloud PostgreSQL

| Provider | Free tier | Notes |
|----------|-----------|-------|
| [Neon](https://neon.tech) | 512 MB, serverless | cold-start may add latency |
| [Supabase](https://supabase.com) | 500 MB | REST + Postgres |
| [Render](https://render.com) | 1 GB (expires 90 days) | always-on within quota |

Set the same `DB_*` secrets in the Colab Secrets panel.

### Option 3: Pipeline-only testing (no DB)

Steps 6–8 prove the full Model A → F pipeline without any database. For full frontend testing with kiosk endpoints, use Option 1 or 2 above.